# Research 04 — 注意力机制手写（SDPA、多头、KV cache）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/research/04-attention.ipynb)

- **日期**：2026-10-02
- **定位**：Research PyTorch 轮次的模型结构面。注意力是唯一一个面试官敢让你从零写的组件 —— 能手写、能和官方实现数值对齐，才算过关。
- **环境**：纯 PyTorch，CPU 即可。

配套笔记：[04-attention.md](04-attention.md)


## 今日目标

1. 手写 scaled dot-product attention 并与 `F.scaled_dot_product_attention` 数值对齐。
2. 手写 causal mask、多头注意力，并与 `nn.MultiheadAttention` 做权重等价验证。
3. 实现 KV cache 的增量解码，验证与全序列前向逐步等价。

公式先对齐：$ \mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V $ 。除 $ \sqrt{d_k} $ 是因为点积方差随 $ d_k $ 线性增长，不缩放 softmax 会饱和、梯度消失。


## 1. 手写 SDPA vs 官方实现

In [ ]:
import torch
import torch.nn.functional as F

def manual_sdpa(Q, K, V, mask=None):
    """手写 scaled dot-product attention。

    Q, K, V: (..., T, d) float —— 本节实际传 (B, h, T, d_k)：B 条、h 个头、
             每头序列长 T、每头维度 d_k；
    mask: 可广播到 (..., T, T) 的加性 mask（禁止位置填 -inf，None = 不 mask），
          在 softmax 之前加到打分上。
    返回: (..., T, d) —— softmax(QK^T / √d_k) V。"""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)   # ← 缩放是 √d_k，不是 d_k
    if mask is not None:
        scores = scores + mask                        # ← mask 在 softmax 之前
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

torch.manual_seed(0)
Q = torch.randn(2, 6, 8); K = torch.randn(2, 6, 8); V = torch.randn(2, 6, 8)
mine = manual_sdpa(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V)
print('SDPA 与官方实现最大差:', (mine - ref).abs().max().item(), '（<1e-6 即对齐）')


## 2. Causal mask：位置 $ t $ 只能看 $ \le t $

In [ ]:
T = 6
mask = torch.full((T, T), float('-inf')).triu(diagonal=1)  # 上三角 -inf
out_mine = manual_sdpa(Q, K, V, mask=mask)
out_ref = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
print('causal 版与官方 is_causal 最大差:', (out_mine - out_ref).abs().max().item())

# 泄漏自检：改最后一个 token 的输入，前 5 个位置的输出必须不变
Q2, K2, V2 = Q.clone(), K.clone(), V.clone()
K2[:, -1] += 100.0; V2[:, -1] += 100.0
out2 = manual_sdpa(Q2, K2, V2, mask=mask)
print('改最后一个 token 后，前 5 个位置输出最大变化:', (out_mine[:, :-1] - out2[:, :-1]).abs().max().item(),
      '（≈0 = 无未来信息泄漏）')


## 3. 多头注意力：从零写，与官方模块等价

$ d_{model} $ 切成 $ h $ 份，每份 $ d_k = d_{model}/h $ ，各做一遍 SDPA 再拼回。维度顺序 (batch, heads, seq, head_dim) 不能错 —— 这是能跑但结果全错的头号坑。


In [ ]:
class MultiHeadAttention(torch.nn.Module):
    """手写多头注意力（拿来和 nn.MultiheadAttention 对拍）。

    __init__: d_model 特征维度（默认 32）；num_heads 头数（默认 4，必须整除 d_model）。
    forward: x (B, T, d_model)；causal=True 时加因果 mask。返回 (B, T, d_model)。"""
    def __init__(self, d_model=32, num_heads=4):
        super().__init__()
        assert d_model % num_heads == 0
        self.h, self.d_k = num_heads, d_model // num_heads
        self.qkv = torch.nn.Linear(d_model, 3 * d_model)   # 一个大投影：工程实现如此
        self.out = torch.nn.Linear(d_model, d_model)

    def forward(self, x, causal=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B, T, D) → (B, h, T, d_k)
        def split(t):
            return t.view(B, T, self.h, self.d_k).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        mask = torch.full((T, T), float('-inf'), device=x.device).triu(1) if causal else None
        y = manual_sdpa(q, k, v, mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, D)   # 拼回 (B, T, D)
        return self.out(y)

torch.manual_seed(0)
mine = MultiHeadAttention(32, 4)
refm = torch.nn.MultiheadAttention(32, 4, batch_first=True, dropout=0.0)
# 把我的权重灌进官方模块：qkv 行块 = [Wq; Wk; Wv]
with torch.no_grad():
    refm.in_proj_weight.copy_(mine.qkv.weight)
    refm.in_proj_bias.copy_(mine.qkv.bias)
    refm.out_proj.weight.copy_(mine.out.weight)
    refm.out_proj.bias.copy_(mine.out.bias)
mine.eval(); refm.eval()
x = torch.randn(2, 5, 32)
with torch.no_grad():
    a = mine(x)
    b, _ = refm(x, x, x)
print('手写 MHA 与 nn.MultiheadAttention 最大差:', (a - b).abs().max().item(), '（<1e-5 即等价）')


## 4. KV cache：增量解码与全序列前向等价

推理时每步只喂一个新 token，K/V 全部缓存复用。验证：逐步解码每一步的输出 == 全前向对应位置的输出。


In [ ]:
# 简化验证：用 SDPA 层面演示 KV cache 等价性 ——
# 第 t 步只算新 token 的 q，k/v 取缓存的前 t+1 个
torch.manual_seed(0)
T, D = 8, 16
x = torch.randn(1, T, D)
Wq, Wk, Wv = [torch.nn.Linear(D, D, bias=False) for _ in range(3)]
Q, K, V = Wq(x), Wk(x), Wv(x)
mask = torch.full((T, T), float('-inf')).triu(1)
full = manual_sdpa(Q, K, V, mask=mask)

k_cache, v_cache = [], []
step_outs = []
for t in range(T):
    k_cache.append(K[:, t:t+1]); v_cache.append(V[:, t:t+1])
    Kcat = torch.cat(k_cache, dim=1); Vcat = torch.cat(v_cache, dim=1)
    step_outs.append(manual_sdpa(Q[:, t:t+1], Kcat, Vcat))   # 无 mask：缓存里只有 ≤t 的位置
cached = torch.cat(step_outs, dim=1)
print('逐步解码（KV cache）与全前向最大差:', (full - cached).abs().max().item(), '（≈0 即等价）')
print('每步 attention 规模: 第 t 步只需算 1 个 query × (t+1) 个 key，全前向是 T×T —— 这就是 KV cache 省的计算')


## 5. Debug 演练：四个可跑的 bug 现场

attention 的 bug 有个共同点：shape 几乎永远合法，报错指望不上，只能靠对拍和自检。下面四个都跑给你看。

In [ ]:
# BUG 1：缩放分母用了 √d_model 而不是 √d_k
def sdpa_wrong_scale(Q, K, V, d_model):
    d = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_model ** 0.5)    # ← BUG：分母应是每头的 d_k
    return torch.softmax(scores, dim=-1) @ V

torch.manual_seed(0)
B, h, T, d_model = 2, 4, 5, 32
d_k = d_model // h
Q = torch.randn(B, h, T, d_k); K = torch.randn(B, h, T, d_k); V = torch.randn(B, h, T, d_k)
ref = torch.nn.functional.scaled_dot_product_attention(Q, K, V)
print('BUG 1 症状: 分母用 √32 的结果与官方最大差 =', round((sdpa_wrong_scale(Q, K, V, d_model) - ref).abs().max().item(), 3), '（正确版 manual_sdpa 是 1e-7 量级）')

**修复 1**：分母永远跟着每个 head 的维度走：`scores / sqrt(d_k)`，其中 `d_k = d_model // num_heads`。

**为什么有效**：缩放是为了让 Q·K 的方差不随维度涨。每个 head 实际做点积的维度是 d_k 不是 d_model；head 越多两者差越远，注意力分布被系统性压平（分母偏大）或压尖。对拍官方实现时这个差立刻现形，所以手写 attention 的第一件事就是留一个对拍 cell。

In [ ]:
# BUG 2：mask 乘在 softmax 之后（还不 renormalize）
def sdpa_mask_after(Q, K, V, keep):
    d = Q.size(-1)
    attn = torch.softmax(Q @ K.transpose(-2, -1) / (d ** 0.5), dim=-1)
    return (attn * keep) @ V                                # ← BUG：权重先成型再砍，未来信息已混进来

torch.manual_seed(0)
x = torch.randn(1, 1, 6, 8)
keep = torch.tril(torch.ones(6, 6)).view(1, 1, 6, 6)
out_bug = sdpa_mask_after(x, x, x, keep)
x2 = x.clone(); x2[0, 0, -1] += 10.0                        # 只改最后一个 token
out_bug2 = sdpa_mask_after(x2, x2, x2, keep)
print('BUG 2 症状: 改最后一个 token 后，前面位置输出最大变化 =', round((out_bug - out_bug2)[0, 0, :-1].abs().max().item(), 4), '（≠0 = 泄漏，正确版自检应为 0）')

**修复 2**：mask 在 softmax **之前**以加法形式进去（禁止位置填 −∞）：`softmax(scores + mask) @ V`，第 1 节的 manual_sdpa 就是这个顺序。

**为什么有效**：softmax 之后再乘 mask，等于先让未来 token 参与归一化再把它们的份额扔掉 —— 信息已经通过分母和加权混进来了，而且每行权重和还不再是 1。顺序就是一切：mask 是打分阶段的裁决，不是输出阶段的补丁。抓它的自检就是第 2 节那个"改最后一个 token"：前面位置输出变化必须严格为 0。

In [ ]:
# BUG 3：拆头时转置错（(B,T,D) 直接 view 成 (B,h,T,d_k)）
def mha_wrong_split(mha, x):
    B, T, D = x.shape
    q, k, v = mha.qkv(x).chunk(3, dim=-1)
    def split_bug(t):
        return t.reshape(B, mha.h, T, mha.d_k)             # ← BUG：少了 view(B,T,h,d_k).transpose(1,2)
    q, k, v = split_bug(q), split_bug(k), split_bug(v)
    y = manual_sdpa(q, k, v)
    return mha.out(y.reshape(B, T, D))

torch.manual_seed(0)
m = MultiHeadAttention(32, 4); m.eval()
xx = torch.randn(2, 5, 32)
with torch.no_grad():
    good = m(xx); bad = mha_wrong_split(m, xx)
print('BUG 3 症状: reshape 下全部 shape 合法、无任何报错，但与正确版最大差 =', round((good - bad).abs().max().item(), 3), '（只有对拍官方实现能抓）')

**修复 3**：拆头固定两步：先 `view(B, T, h, d_k)` 再 `transpose(1, 2)`（第 3 节的 split 就是）；合并时逆操作 `transpose(1, 2).contiguous().view(B, T, D)`。

**为什么有效**：(B, T, D) 里 D 的排布是"每个时间步的完整特征"，错位拼进 (B, h, T, d_k) 等于把不同时间步的数塞进同一个 head —— 语义全毁。用 view 写错时 PyTorch 多半直接报错（那是幸运版）；用 reshape 或输入恰好 contiguous 时它一声不吭地算出一堆垃圾，且维度数字全凑得齐，shape 检查永远沉默。这类 bug 唯一的抓法就是和官方实现逐元素对拍，手写 attention 必须把对拍写进日常而不是出事后。

In [ ]:
# BUG 4：padding 行整行被 mask，-1e9 顶替 -inf 时漏出均匀权重
torch.manual_seed(0)
scores = torch.zeros(1, 4)                                  # 一个 query 对 4 个 key 的打分
mask_bug = torch.tensor([0., -1e9, -1e9, -1e9])             # 只准看第 0 个 key，其余用 -1e9 顶替
print('BUG 4 症状: 普通行上 -1e9 看起来完全正常 =', torch.softmax(scores + mask_bug, dim=-1).tolist()[0], '—— 这就是它藏得深的原因')
scores_row = torch.full((1, 4), -1e9)                       # padding 位置的 query 行：整行都被禁
print('         整行被禁时权重变均匀分布 =', torch.softmax(scores_row, dim=-1).tolist()[0], '（它在"认真地"平均看所有 key）')
mask_fix = torch.tensor([0., float('-inf'), float('-inf'), float('-inf')])
print('         -inf 版普通行 =', torch.softmax(scores + mask_fix, dim=-1).tolist()[0], '（被禁 key 严格 0；整行全禁会 NaN，靠 loss 端丢弃 padding 行兜底）')

**修复 4**：两条一起做：(1) mask 值用 `-inf`（或该 dtype 的最小有限值），被禁 key 的权重严格为 0；(2) padding 位置的 query 行在实践中不靠 mask 兜底——它的输出在 loss 计算时用 label 的 -100 直接丢弃。

**为什么有效**：`-1e9` 是有限数，softmax 里 exp(−1e9 − max) 在整行都是 −1e9 时等于 exp(0)=1，于是"全员禁止"被算成"均匀出席"，padding 行会输出一个看起来正常的假向量，还可能经残差流进真实位置。用 −inf 后正常行严格置零；整行 NaN 的风险则由"padding 行输出必被丢弃"这条工程纪律兜住，而不是指望 mask 本身。

## 6. Debug 速查（面试给代码找 bug 时的对号入座）

修复都在第 5 节各 bug 正下方，这里只留对号顺序：缩放分母是 d_k 不是 d_model（BUG 1）→ mask 在 softmax 之前（BUG 2）→ 拆头先 view 再 transpose（BUG 3）→ mask 用 −inf 且 padding 行靠 loss mask 丢弃（BUG 4）。

## 小结

- SDPA 三件套：投影、$ 1/\sqrt{d_k} $ 缩放、softmax 前的 mask。
- 多头的命门是维度顺序，对拍官方实现是手写正确性的唯一标准。
- KV cache 与全前向严格等价，每步复杂度从 $ O(T^2) $ 降到 $ O(T) $ ，代价是显存随 $ T $ 线性涨。

**Research 系列完**（01 训练循环 → 02 DPO → 03 PPO/GRPO → 04 注意力）。下一条线：并行基础（TP 手写 linear、ZeRO/FSDP），Day 01 的 DDP 在那条线等着。
